In [1]:
import pandas as pd

import set_paths
from config.paths import *
from config.config import columnas_archivo_citas
from src.utils import get_files, strip_df
from src.preprocessing import load_citas_df

In [2]:
crm_path = get_files(CRM)
crm_df = pd.read_csv(CRM/ crm_path[0])

In [ ]:
total_leads = crm_df[["ID Lead", 
                      "a quien fue traspasado"]].groupby(by = "a quien fue traspasado", 
                                                         as_index = False, 
                                                         observed = False).count().sort_values(by="ID Lead", 
                                                                                               ascending=False, 
                                                                                               ignore_index = True)
strip_df(total_leads)
total_leads.columns = ["asesor", "total_leads"]
total_leads

,asesor,total_leads
0,Leonardo Furlong,132
1,Diana Elizabeth Vivanco Galindo,42
2,Alejandra Aurora Corona,41
3,Jose Manuel Alvarez Hernandez,29
4,Victor Manuel Garcia Bautista,29
5,Jaqueline Jiménez,26
6,Daniel Guerra Almazan,21
7,Yolanda Rojas Aguilar,20
8,Valentina Mariel Salgado Arias,16
9,Patricia Nathalie Mijares Perez,13


In [4]:

def desglose_citas_asesor(citas_df: pd.DataFrame, 
                          asesores: list[str]):

    test = citas_df.groupby(by = ["ASESOR", "ASISTENCIA"], 
                            as_index= False, 
                            observed = True).count().sort_values(by = "ASESOR", 
                                                                 ascending = False, 
                                                                 ignore_index = True).iloc[:, :3]

    desglose = {"atendida": {asesor: 0 for asesor in asesores},
                "reagendada" : {asesor: 0 for asesor in asesores},
                "cancelada": {asesor: 0 for asesor in asesores},
                "otros": {asesor: 0 for asesor in asesores}
                }
    estados = ["atendida", "reagendada", "cancelada"]

    for asesor in asesores:
        total_citas = test[test["ASESOR"] == asesor].loc[:, "ID"].sum()

        for status_cita in estados:

            resultado = test.loc[
                (test["ASESOR"] == asesor)
                & (test["ASISTENCIA"] == status_cita),
                "ID"
            ]

            if not resultado.empty:
                desglose[status_cita][asesor] = resultado.iloc[0]
            else:
                desglose[status_cita][asesor] = 0

        total_conocidos = sum(
        desglose[estado][asesor]
        for estado in estados
    )

        desglose["otros"][asesor] = (
        total_citas - total_conocidos
    )

    res_citas = pd.DataFrame(desglose)
    res_citas["total"] = res_citas.sum(axis = 1)


    return test, res_citas

In [5]:
citas = load_citas_df(CITAS / get_files(CITAS)[0])

In [7]:
test, test_2 = desglose_citas_asesor(citas_df= citas, asesores = citas["ASESOR"].unique())

In [8]:
test_2

,atendida,reagendada,cancelada,otros,total
Valentina Salgado,2,0,1,0,3
Vanessa Arias,0,0,1,0,1
Diana Vivanco,2,2,2,0,6
Alejandra Corona,1,0,1,0,2
Daniel Guerra,2,1,0,0,3
Arturo Gonzalez,1,0,0,0,1
Jose Manuel Alvarez,1,0,1,1,3
Victor Garcia,1,1,0,0,2
